# Real Option-Chain Implied-Volatility Snapshot

This notebook applies the project's implied-volatility tools to a user-supplied option-chain snapshot. A real CSV is not committed because market-data redistribution rights depend on the provider. The workflow is deliberately source-agnostic.

## Expected input

Place a permitted option-chain export at `data/raw/option_chain_snapshot.csv`. The expected schema is documented in `data/raw/README.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data" / "raw" / "option_chain_snapshot.csv"
DATA_PATH

In [ ]:
import pandas as pd

if DATA_PATH.exists():
    raw_chain = pd.read_csv(DATA_PATH)
    display(raw_chain.head())
    print(f"rows: {len(raw_chain):,}")
else:
    raw_chain = None
    print("No option_chain_snapshot.csv found. Add a permitted market-data snapshot to data/raw/ and rerun this notebook.")

In [ ]:
from options_lab.market_data import OptionChainCleaningConfig, recover_market_implied_volatilities

if raw_chain is not None:
    config = OptionChainCleaningConfig(maximum_relative_spread=0.50, minimum_time_to_expiry=7/365, minimum_vega=1e-4)
    cleaned = recover_market_implied_volatilities(raw_chain, config)
    display(cleaned[["expiry", "option_type", "strike", "mid", "relative_spread", "time_to_expiry", "recovered_implied_volatility", "model_vega"]].head())
    print(f"contracts retained after cleaning: {len(cleaned):,}")
else:
    cleaned = None

In [ ]:
import matplotlib.pyplot as plt

if cleaned is not None and not cleaned.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for expiry, group in cleaned.sort_values("expiry").groupby("expiry"):
        ax.scatter(group["log_forward_moneyness"], 100 * group["recovered_implied_volatility"], s=18, label=str(expiry.date()))
    ax.axvline(0.0, linewidth=1)
    ax.set_xlabel("Log-forward moneyness: ln(K/F)")
    ax.set_ylabel("Recovered implied volatility (%)")
    ax.set_title("Market option-chain implied-volatility smile/skew")
    ax.legend(fontsize=8)
    plt.show()

## Interpretation checklist

When a real snapshot is loaded, inspect: wide bid-ask spreads, low-vega contracts, failed IV inversions, missing strikes, call-put inconsistencies, and whether the observed smile/skew differs materially from the synthetic surface used in the controlled notebooks.